# Stylepal Agent Test Harness

Runs the Stylepal Stylist Agent against test cases and scores with Ragas metrics:
- **Agent Goal Accuracy**: Did the agent achieve the user's goal? (LLM-as-judge)
- **Topic Adherence**: Does the agent stay on topic?
- **Tool Coverage**: Fraction of expected tools called at least once (subset semantics; works for multi-step agents).

Use `AGENT_TEST_QUERIES` for original 6 cases, or `AGENT_TEST_QUERIES_EXTENDED` for 14 cases.

## Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "eval":
    project_root = project_root.parent
backend_dir = project_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")
_ = load_dotenv(backend_dir / ".env")

In [2]:
import asyncio
import json
import os
import time
import warnings
warnings.filterwarnings("ignore", message=".*google.generativeai.*", category=FutureWarning)

from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from ragas.dataset_schema import MultiTurnSample
from ragas.integrations.langgraph import convert_to_ragas_messages
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import AgentGoalAccuracyWithReference, TopicAdherenceScore
from ragas.messages import ToolCall

from services.agent import get_stylist_graph

EVAL_DELAY_SEC = 60  # Wait between evals to avoid 429 rate limit

/Users/sireeshapulipati/stylepal/venv/lib/python3.12/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_4168/2139144194.py:13: DeprecationWarning: Importing AgentGoalAccuracyWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AgentGoalAccuracyWithReference
  from ragas.metrics import AgentGoalAccuracyWithReference, TopicAdherenceScore
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_4168/2139144194.py:13: DeprecationW

## Test Cases

In [3]:
AGENT_TEST_QUERIES = [
    {
        "query": "What should I wear for a job interview tomorrow?",
        "reference": "Suggest two professional outfit options from the wardrobe for a job interview.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "job interview professional outfit"}),
            ToolCall(name="get_weather", args={"question": "What should I wear for a job interview tomorrow?"}),
        ],
    },
    {
        "query": "What necklines work for an hourglass body type?",
        "reference": "Explain which necklines flatter an hourglass figure based on style principles.",
        "reference_tool_calls": [ToolCall(name="retrieve_style_knowledge", args={"query": "necklines hourglass body type"})],
    },
    {
        "query": "Outfit for a casual Friday at the office",
        "reference": "Suggest two outfit options suitable for casual Friday from the wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "casual Friday office outfit"}),
        ],
    },
    {
        "query": "I moved to NYC",
        "reference": "Update the user's profile with their new location.",
        "reference_tool_calls": [ToolCall(name="update_profile", args={"location": "NYC"})],
    },
    {
        "query": "I bought a navy blazer",
        "reference": "Add the new navy blazer to the user's wardrobe.",
        "reference_tool_calls": [ToolCall(name="add_wardrobe_item", args={"name": "navy blazer", "category": "outerwear", "color": "navy"})],
    },
    {
        "query": "I don't wear item 5 anymore, remove it from my wardrobe",
        "reference": "Soft delete item 5 so it no longer appears in suggestions.",
        "reference_tool_calls": [ToolCall(name="deprecate_wardrobe_item", args={"item_id": 5})],
    },
]

In [4]:
AGENT_TEST_QUERIES_EXTENDED = [
    {
        "query": "What to wear for a dinner date this Saturday?",
        "reference": "Suggest two outfit options suitable for a dinner date from the wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "dinner date outfit"}),
            ToolCall(name="get_weather", args={"question": "What to wear for a dinner date this Saturday?"}),
        ],
    },
    {
        "query": "Interview outfit for next Tuesday",
        "reference": "Suggest two professional outfit options for a job interview.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "job interview professional outfit"}),
        ],
    },
    {
        "query": "Plan outfits for my 3-day trip to Paris July 15-17: Day 1 travel, Day 2 museum and casual dinner, Day 3 business meeting",
        "reference": "Suggest one primary outfit per day/occasion, optionally 1-3 extra items to pack.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "Paris July 15-17", "location": "Paris"}),
        ],
    },
    {
        "query": "5-day business trip to NYC Mon-Fri: meetings all day, one formal dinner Wednesday evening",
        "reference": "Suggest one outfit per day/occasion for the business trip.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "NYC weather forecast", "location": "NYC"}),
        ],
    },
    {
        "query": "Outfit for a local tech conference I'm attending next Tuesday, business casual",
        "reference": "Suggest two outfit options suitable for a tech conference.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="retrieve_style_knowledge", args={"query": "business casual conference outfit"}),
            ToolCall(name="get_weather", args={"question": "weather next Tuesday"}),
        ],
    },
    {
        "query": "I prefer tailored fits now",
        "reference": "Update the user's profile with their new style preferences.",
        "reference_tool_calls": [ToolCall(name="update_profile", args={"silhouette_preferences": "tailored"})],
    },
    {
        "query": "I'm 35 and my body type is more pear-shaped",
        "reference": "Update the user's profile with age and body type.",
        "reference_tool_calls": [ToolCall(name="update_profile", args={"age": 35, "body_type": "pear"})],
    },
    {
        "query": "I bought a navy blazer from Ralph Lauren last month for work",
        "reference": "Add the navy blazer to the wardrobe with brand, purchase date, and occasion.",
        "reference_tool_calls": [
            ToolCall(name="add_wardrobe_item", args={"name": "navy blazer", "category": "outerwear", "color": "navy", "brand": "Ralph Lauren", "purchased_at": "last month", "occasion_tags": "work"})
        ],
    },
    {
        "query": "Add my new white sneakers from Nike, bought in March 2024, casual wear",
        "reference": "Add the white sneakers to the wardrobe.",
        "reference_tool_calls": [
            ToolCall(name="add_wardrobe_item", args={"name": "white sneakers", "category": "shoes", "color": "white", "brand": "Nike", "purchased_at": "March 2024", "occasion_tags": "casual"})
        ],
    },
    {
        "query": "Remove item 12 from my wardrobe",
        "reference": "Soft delete item 12 so it no longer appears in suggestions.",
        "reference_tool_calls": [ToolCall(name="deprecate_wardrobe_item", args={"item_id": 12})],
    },
    {
        "query": "What colors pair well with navy?",
        "reference": "Explain which colors complement navy based on style principles.",
        "reference_tool_calls": [ToolCall(name="retrieve_style_knowledge", args={"query": "colors pair navy"})],
    },
    {
        "query": "What's the golden mean in fashion?",
        "reference": "Explain the golden mean concept in fashion.",
        "reference_tool_calls": [ToolCall(name="retrieve_style_knowledge", args={"query": "golden mean fashion proportions"})],
    },
    {
        "query": "Outfit for tomorrow, it might rain",
        "reference": "Suggest outfit options considering rain, with weather check.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "tomorrow weather rain"}),
        ],
    },
    {
        "query": "What to wear in Paris next week?",
        "reference": "Suggest outfit options for Paris with weather consideration.",
        "reference_tool_calls": [
            ToolCall(name="get_wardrobe", args={}),
            ToolCall(name="get_weather", args={"question": "Paris next week", "location": "Paris"}),
        ],
    },
]

## Harness: Run Agent + Ragas Scorers

In [5]:
def run_agent_and_get_messages(query: str, thread_id: str | None = None) -> list:
    """Run agent and return messages for Ragas conversion. Uses unique thread_id per run."""
    import uuid
    graph = get_stylist_graph()
    tid = thread_id or f"eval-{uuid.uuid4().hex[:8]}"
    config = {
        "configurable": {"thread_id": tid},
        "run_name": "StylepalAgent",
        "tags": ["stylepal", "stylist", "eval"],
        "metadata": {"thread_id": tid, "eval": True},
        "recursion_limit": 12,
    }
    result = graph.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config,
    )
    return result["messages"]


def ensure_tool_calls_for_ragas(messages):
    """Ragas reads tool_calls from additional_kwargs (OpenAI format). Gemini uses message.tool_calls."""
    out = []
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            tc = m.tool_calls
            if tc:
                def _to_openai(t, i):
                    n = t.get("name", "") if isinstance(t, dict) else getattr(t, "name", "")
                    a = t.get("args", {}) if isinstance(t, dict) else getattr(t, "args", {})
                    a = a if isinstance(a, dict) else {}
                    return {"id": t.get("id", f"call_{i}") if isinstance(t, dict) else getattr(t, "id", f"call_{i}"), "type": "function", "function": {"name": n, "arguments": json.dumps(a)}}
                openai_tc = [_to_openai(t, i) for i, t in enumerate(tc)]
                kwargs = dict(getattr(m, "additional_kwargs", None) or {})
                kwargs["tool_calls"] = openai_tc
                out.append(AIMessage(content=m.content, additional_kwargs=kwargs))
                continue
        out.append(m)
    return out


def tool_coverage_score(messages: list, expected_tool_names: list) -> float:
    """Subset-based: fraction of expected tools called at least once. Returns 1.0 if none expected."""
    if not expected_tool_names:
        return 1.0
    called = set()
    for m in messages:
        if isinstance(m, AIMessage) and getattr(m, "tool_calls", None):
            for t in m.tool_calls:
                n = t.get("name", "") if isinstance(t, dict) else getattr(t, "name", "")
                if n:
                    called.add(n)
    expected = set(expected_tool_names)
    return len(expected & called) / len(expected)

In [6]:
REFERENCE_TOPICS = [
    "wardrobe", "styling", "outfit suggestions", "fashion", "professional dress", "clothing",
    "profile updates", "user preferences",  # Include for profile-update queries
]

evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
)

goal_scorer = AgentGoalAccuracyWithReference()
goal_scorer.llm = evaluator_llm
topic_scorer = TopicAdherenceScore(llm=evaluator_llm, mode="precision")

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_4168/1420482000.py:6: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


In [7]:
async def run_eval_loop(samples: list, label: str = "Eval"):
    scores_goal, scores_topic, scores_tool_call = [], [], []
    for i, sample in enumerate(samples):
        if i > 0:
            print(f"Waiting {EVAL_DELAY_SEC}s before next eval...")
            await asyncio.sleep(EVAL_DELAY_SEC)
        print(f"{label} {i+1}/{len(samples)}: {sample['query'][:50]}...")
        t0 = time.perf_counter()
        messages_raw = run_agent_and_get_messages(sample["query"])
        t_agent = time.perf_counter() - t0
        print(f"  Agent done in {t_agent:.1f}s. Running Ragas scorers...", flush=True)
        expected_tool_names = [t.name if hasattr(t, "name") else t.get("name", "") for t in sample.get("reference_tool_calls", [])]
        tool_cov = tool_coverage_score(messages_raw, expected_tool_names)
        scores_tool_call.append(tool_cov)
        messages = ensure_tool_calls_for_ragas(messages_raw)
        ragas_trace = convert_to_ragas_messages(messages)

        mt_sample = MultiTurnSample(user_input=ragas_trace, reference=sample["reference"])
        goal_score = await goal_scorer.multi_turn_ascore(mt_sample)
        scores_goal.append(goal_score)

        mt_sample_ta = MultiTurnSample(user_input=ragas_trace, reference_topics=REFERENCE_TOPICS)
        topic_score = await topic_scorer.multi_turn_ascore(mt_sample_ta)
        scores_topic.append(topic_score)

        t_total = time.perf_counter() - t0
        print(f"  Goal: {goal_score:.4f} | Topic: {topic_score:.4f} | ToolCov: {tool_cov:.4f} | Total: {t_total:.1f}s")
        print()

    return scores_goal, scores_topic, scores_tool_call

## Run Evaluation

In [8]:
# Choose: AGENT_TEST_QUERIES (6) or AGENT_TEST_QUERIES_EXTENDED (14)
SAMPLES = AGENT_TEST_QUERIES  # or AGENT_TEST_QUERIES_EXTENDED

scores_goal, scores_topic, scores_tool_call = await run_eval_loop(SAMPLES, label="Eval")

Eval 1/6: What should I wear for a job interview tomorrow?...
  Agent done in 74.9s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.7143 | ToolCov: 1.0000 | Total: 93.2s

Waiting 60s before next eval...
Eval 2/6: What necklines work for an hourglass body type?...
  Agent done in 167.4s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.0000 | ToolCov: 0.0000 | Total: 175.0s

Waiting 60s before next eval...
Eval 3/6: Outfit for a casual Friday at the office...
  Agent done in 34.5s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.3333 | ToolCov: 1.0000 | Total: 45.7s

Waiting 60s before next eval...
Eval 4/6: I moved to NYC...
  Agent done in 229.7s. Running Ragas scorers...
  Goal: 1.0000 | Topic: 0.0000 | ToolCov: 1.0000 | Total: 236.0s

Waiting 60s before next eval...
Eval 5/6: I bought a navy blazer...
  Agent done in 21.9s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 1.0000 | ToolCov: 1.0000 | Total: 28.7s

Waiting 60s before next eval...
Eval 6/6: I don't wear item 5 

In [9]:
import numpy as np
print("Evaluation Summary:")
print(f"  Mean Agent Goal Accuracy: {np.mean(scores_goal):.4f}")
print(f"  Mean Topic Adherence: {np.mean(scores_topic):.4f}")
print(f"  Mean Tool Coverage: {np.mean(scores_tool_call):.4f}")

Evaluation Summary:
  Mean Agent Goal Accuracy: 0.1667
  Mean Topic Adherence: 0.4079
  Mean Tool Coverage: 0.8333


In [10]:
# Choose: AGENT_TEST_QUERIES (6) or AGENT_TEST_QUERIES_EXTENDED (14)
SAMPLES = AGENT_TEST_QUERIES_EXTENDED

scores_goal, scores_topic, scores_tool_call = await run_eval_loop(SAMPLES, label="Eval")

Eval 1/14: What to wear for a dinner date this Saturday?...
  Agent done in 43.0s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.3333 | ToolCov: 1.0000 | Total: 68.5s

Waiting 60s before next eval...
Eval 2/14: Interview outfit for next Tuesday...
  Agent done in 35.8s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.5000 | ToolCov: 1.0000 | Total: 68.1s

Waiting 60s before next eval...
Eval 3/14: Plan outfits for my 3-day trip to Paris July 15-17...
  Agent done in 42.6s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.8000 | ToolCov: 1.0000 | Total: 106.5s

Waiting 60s before next eval...
Eval 4/14: 5-day business trip to NYC Mon-Fri: meetings all d...
  Agent done in 39.8s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.4286 | ToolCov: 1.0000 | Total: 63.2s

Waiting 60s before next eval...
Eval 5/14: Outfit for a local tech conference I'm attending n...
  Agent done in 185.3s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.0000 | ToolCov: 0.0000 | Total: 198.7s

W

#### Extended Evaluation Summary:
  
  - Mean Agent Goal Accuracy: 0.1429
  - Mean Topic Adherence: 0.3211
  - Mean Tool Coverage: 0.8571

## Conclusions: Original vs Extended Evaluation

### Summary

| Metric | Original (6 samples) | Extended (14 samples) |
|--------|----------------------|------------------------|
| **Agent Goal Accuracy** | ~0.17–0.33 | ~0.07–0.14 |
| **Topic Adherence** | ~0.41 | ~0.32–0.48 |
| **Tool Coverage** | ~0.83 | ~0.86 |

- **Original:** 1–2 passed (e.g. "I moved to NYC", "Remove item 5").
- **Extended:** 1–2 passed (e.g. "I'm 35 and pear-shaped", "Add white sneakers").
- **Tool Coverage** is high: the agent usually calls the expected tools.
- **Topic Adherence** is moderate and inconsistent across query types.


### Why Goal Accuracy Is So Low

Agent Goal Accuracy uses an LLM judge that compares the inferred `end_state` of the conversation to a reference outcome. It returns 0 or 1 with no partial credit.

**Possible causes of low scores:**

1. **Strict comparison** – The judge treats the inferred outcome as different from the reference unless they match closely.

2. **Format mismatch** – The agent may produce valid responses (e.g. two outfit options) but in a different structure (e.g. "Option A/B" vs "OUTFIT 1/2"), so the judge may mark it as a mismatch.

3. **Clarifying questions** – If the agent asks follow-up questions before suggesting outfits, the judge may treat the goal as not yet achieved.

4. **Evaluator vs agent size** – The evaluator (gpt-4o-mini) may be smaller than the agent (gemini-2.5-pro), which can lead to judge errors or inconsistent verdicts.

5. **Narrow references** – Each reference is a single short string. Multiple valid outcomes or phrasing are not captured.

6. **False negatives** – Agent behavior may be correct but still scored 0 due to semantic or phrasing differences.

**Recommendation:** Use a stronger evaluator (e.g. gpt-4.1) when available, and relax or broaden references to reduce false negatives.

### Optional: Run evals with stronger evaluator (gpt-4.1)

Run this cell to switch the Ragas evaluator to gpt-4.1, then run the eval loop cell above to re-score with the new model. The agent will run again; only the judge model changes.

In [ ]:
# Switch evaluator to gpt-4.1 (run this, then run the eval loop cell)
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4.1", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
)
goal_scorer.llm = evaluator_llm
topic_scorer = TopicAdherenceScore(llm=evaluator_llm, mode="precision")
print("Evaluator switched to gpt-4.1. Run the eval loop cell to score with this model.")

Evaluator switched to gpt-4.1. Run the eval loop cell to score with this model.


/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_4168/1565917006.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(


In [16]:
# Choose: AGENT_TEST_QUERIES (6) or AGENT_TEST_QUERIES_EXTENDED (14)
SAMPLES = AGENT_TEST_QUERIES

scores_goal, scores_topic, scores_tool_call = await run_eval_loop(SAMPLES, label="Eval")

Eval 1/6: What should I wear for a job interview tomorrow?...
  Agent done in 38.8s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 1.0000 | ToolCov: 1.0000 | Total: 117.9s

Waiting 60s before next eval...
Eval 2/6: What necklines work for an hourglass body type?...
  Agent done in 166.3s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.0000 | ToolCov: 0.0000 | Total: 174.5s

Waiting 60s before next eval...
Eval 3/6: Outfit for a casual Friday at the office...
  Agent done in 45.9s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.6667 | ToolCov: 1.0000 | Total: 92.2s

Waiting 60s before next eval...
Eval 4/6: I moved to NYC...
  Agent done in 168.7s. Running Ragas scorers...
  Goal: 1.0000 | Topic: 0.0000 | ToolCov: 1.0000 | Total: 176.1s

Waiting 60s before next eval...
Eval 5/6: I bought a navy blazer...
  Agent done in 30.1s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 1.0000 | ToolCov: 1.0000 | Total: 38.0s

Waiting 60s before next eval...
Eval 6/6: I don't wear item 5

## Conclusions: gpt-4o-mini vs gpt-4.1 Evaluator

### Summary (All Results)

| Metric | Original (6) gpt-4o-mini | Original (6) gpt-4.1 | Extended (14) gpt-4o-mini | Extended (14) gpt-4.1 |
|--------|-------------------------|----------------------|---------------------------|------------------------|
| **Agent Goal Accuracy** | 0.1667 | 0.3333 | 0.1429 | 0.5714 |
| **Topic Adherence** | 0.4079 | 0.6111 | 0.3211 | 0.7967 |
| **Tool Coverage** | 0.8333 | 0.8333 | 0.8571 | 0.8571 |

### gpt-4.1 Impact

- **Goal Accuracy:** +100% on original (0.17 → 0.33), +300% on extended (0.14 → 0.57). 8/14 extended cases passed with gpt-4.1 vs 2/14 with gpt-4o-mini.
- **Topic Adherence:** +50% on original (0.41 → 0.61), +148% on extended (0.32 → 0.80).
- **Tool Coverage:** Unchanged (deterministic metric; not LLM-based).

### Interpretation

The stronger evaluator (gpt-4.1) scores the same agent much higher. This suggests many prior "failures" were **false negatives** from the weaker gpt-4o-mini judge—the agent was often correct but the judge was too strict or inconsistent. Using a stronger evaluator reduces false negatives and gives a more accurate picture of agent performance.

In [17]:
import numpy as np
print("Evaluation Summary:")
print(f"  Mean Agent Goal Accuracy: {np.mean(scores_goal):.4f}")
print(f"  Mean Topic Adherence: {np.mean(scores_topic):.4f}")
print(f"  Mean Tool Coverage: {np.mean(scores_tool_call):.4f}")

Evaluation Summary:
  Mean Agent Goal Accuracy: 0.3333
  Mean Topic Adherence: 0.6111
  Mean Tool Coverage: 0.8333


In [18]:
# Choose: AGENT_TEST_QUERIES (6) or AGENT_TEST_QUERIES_EXTENDED (14)
SAMPLES = AGENT_TEST_QUERIES_EXTENDED

scores_goal, scores_topic, scores_tool_call = await run_eval_loop(SAMPLES, label="Eval")

Eval 1/14: What to wear for a dinner date this Saturday?...
  Agent done in 42.4s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.8000 | ToolCov: 1.0000 | Total: 128.7s

Waiting 60s before next eval...
Eval 2/14: Interview outfit for next Tuesday...
  Agent done in 35.5s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.7143 | ToolCov: 1.0000 | Total: 161.8s

Waiting 60s before next eval...
Eval 3/14: Plan outfits for my 3-day trip to Paris July 15-17...
  Agent done in 40.9s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 0.7500 | ToolCov: 1.0000 | Total: 279.8s

Waiting 60s before next eval...
Eval 4/14: 5-day business trip to NYC Mon-Fri: meetings all d...
  Agent done in 34.5s. Running Ragas scorers...
  Goal: 1.0000 | Topic: 0.6667 | ToolCov: 1.0000 | Total: 184.9s

Waiting 60s before next eval...
Eval 5/14: Outfit for a local tech conference I'm attending n...
  Agent done in 187.2s. Running Ragas scorers...
  Goal: 0.0000 | Topic: 1.0000 | ToolCov: 0.0000 | Total: 196.4s

In [19]:
import numpy as np
print("Evaluation Summary:")
print(f"  Mean Agent Goal Accuracy: {np.mean(scores_goal):.4f}")
print(f"  Mean Topic Adherence: {np.mean(scores_topic):.4f}")
print(f"  Mean Tool Coverage: {np.mean(scores_tool_call):.4f}")

Evaluation Summary:
  Mean Agent Goal Accuracy: 0.5714
  Mean Topic Adherence: 0.7967
  Mean Tool Coverage: 0.8571
